# Simple Customer Service Agent
_Building a simple customer service agent._

This minimal customer service agent supports only the following service requests from its customers:
1. Cancel order
2. Update delivery address
3. Issue refund

Each request is handled independently, and required parameters are always collected fresh for that request.

In [ ]:
# Imports packages

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama.chat_models import ChatOllama

In [2]:
# Sets endpoints for Ollama models to be available over web requests.
 
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

## Tools

Uses decorator to convert Python functions to LangChain tools that the agent will be using.

These tools are dummy for this lab. experiment. But these will have real implementation for production systems.

In [3]:
@tool
def cancel_order(order_id: str) -> str:
    """Cancels an existing order using order_id."""
    return f"[SUCCESS] Order {order_id} has been cancelled."

@tool
def update_delivery_address(order_id: str, new_delivery_address: str) -> str:
    """Updates delivery address for an existing order."""
    return f"[SUCCESS] Delivery address was updated for order {order_id}. New address is {new_delivery_address}"

@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund for the given order."""
    return f"[SUCCESS] Refund for amount {amount} was issued for order id {order_id}."

tools = {
    "cancel_order": cancel_order,
    "update_delivery_address": update_delivery_address,
    "issue_refund": issue_refund
}

In [4]:
agent_instructions = """You are a customer service agent for an eCommerce platform.
You can only help with three services:
1) cancel order
2) update delivery address
3) issue refund

Rules:
- Keep responses brief and clear.
- If a request is outside these three services, politely decline and restate supported services.
- Never invent order_id or new_delivery_address.
- Do not call tools until all required fields are explicitly provided by the customer.
- For every new request, fetch parameter values from the customer again. Do not reuse values from prior turns.

For cancel order:
- Required field: order_id
- If order_id is missing, ask only for order_id.
- If order_id is present, call cancel_order(order_id) directly.

For update delivery address:
- Required fields: order_id and new_delivery_address
- If any is missing, ask follow-up for missing fields only.
- If both are present, call update_delivery_address(order_id, new_delivery_address) directly.

For issue refund:
- Required fields: order_id and amount
- If the amount is not available, default to $100.
- Call issue_refund(order_id, amount) directly.

After tool execution:
- Show the tool confirmation message to the customer.
"""

## Chat Model

In [ ]:
# Initializes chat client
client = ChatOllama(model=OLLAMA_MODEL, 
                    reasoning=False,
                    base_url=OLLAMA_ENDPOINT
                    )

# Binds to the tools with the chat client
client_with_tools = client.bind_tools([cancel_order, update_delivery_address])

In [6]:
# Maintains a list of all messages that flow across interactions
messages = [
    SystemMessage(content=agent_instructions)   # Initializes with intruction (as a SystemMessage) that is specific to agent.
    ]

In [7]:
display(messages)        # Checks structure of first message i.e. SystemMessage

[SystemMessage(content='You are a customer service agent for an eCommerce platform.\nYou can only help with three services:\n1) cancel order\n2) update delivery address\n3) issue refund\n\nRules:\n- Keep responses brief and clear.\n- If a request is outside these three services, politely decline and restate supported services.\n- Never invent order_id or new_delivery_address.\n- Do not call tools until all required fields are explicitly provided by the customer.\n- For every new request, fetch parameter values from the customer again. Do not reuse values from prior turns.\n\nFor cancel order:\n- Required field: order_id\n- If order_id is missing, ask only for order_id.\n- If order_id is present, call cancel_order(order_id) directly.\n\nFor update delivery address:\n- Required fields: order_id and new_delivery_address\n- If any is missing, ask follow-up for missing fields only.\n- If both are present, call update_delivery_address(order_id, new_delivery_address) directly.\n\nFor issue re

In [8]:
query = "I want to cancel my order 'Order_12345'."      # Customer query to be sent to client
messages.append(HumanMessage(query))                    # Also appends customer query (as a HumanMessage)

In [9]:
display(messages)       # Checks structure of second message i.e. HumanMessage

[SystemMessage(content='You are a customer service agent for an eCommerce platform.\nYou can only help with three services:\n1) cancel order\n2) update delivery address\n3) issue refund\n\nRules:\n- Keep responses brief and clear.\n- If a request is outside these three services, politely decline and restate supported services.\n- Never invent order_id or new_delivery_address.\n- Do not call tools until all required fields are explicitly provided by the customer.\n- For every new request, fetch parameter values from the customer again. Do not reuse values from prior turns.\n\nFor cancel order:\n- Required field: order_id\n- If order_id is missing, ask only for order_id.\n- If order_id is present, call cancel_order(order_id) directly.\n\nFor update delivery address:\n- Required fields: order_id and new_delivery_address\n- If any is missing, ask follow-up for missing fields only.\n- If both are present, call update_delivery_address(order_id, new_delivery_address) directly.\n\nFor issue re

In [10]:
# Passes messages (as a single input) to model and receives response
client_message = client_with_tools.invoke(messages)

In [11]:
# Displays the (structured) model-returned message to analyze element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those later
display(client_message)

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-04T01:37:51.492688686Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20925967159, 'load_duration': 3665283337, 'prompt_eval_count': 459, 'prompt_eval_duration': 15487554276, 'eval_count': 21, 'eval_duration': 1745933508, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019e9047-45c1-75f1-9beb-b6a053d91aef-0', tool_calls=[{'name': 'cancel_order', 'args': {'order_id': 'Order_12345'}, 'id': '4970f402-2d0c-497f-b89c-eb1c4b7ae562', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 459, 'output_tokens': 21, 'total_tokens': 480})

In [12]:
messages.append(client_message)         # Appends last received message into message list

In [ ]:
# Now, client loops over each tool name to call the tool manually for execution
for tool_call in client_message.tool_calls:
    tool_name = tool_call["name"].lower()               # Extracts tool name and converts to lower case
    selected_tool = tools[tool_name]                    # Gets pointer to tool
    tool_message = selected_tool.invoke(tool_call)      # Invokes (or calls) the tool with arguments
    
    # Prints tool name, its arguments and its output for analysis
    print(f"Tool: '{tool_call["name"]}', \nArguments: {tool_call["args"]}, \nTool Output: {tool_message.content}")
    
    messages.append(tool_message)                       # Message received from tool gets added to message list

final_response = client_with_tools.invoke(messages)         # Passes all messages back to model to generate final response
print(f"Final Agent Response: {final_response.content}")    # Prints final response that the customer would see

Tool: 'cancel_order', 
Arguments: {'order_id': 'Order_12345'}, 
Tool Output: [SUCCESS] Order Order_12345 has been cancelled.
Final Agent Response: Thank you for confirming that your order has been cancelled successfully! If you have any other questions or concerns, feel free to ask.


In [14]:
# Displays the final message list for analysis
display(messages)

[SystemMessage(content='You are a customer service agent for an eCommerce platform.\nYou can only help with three services:\n1) cancel order\n2) update delivery address\n3) issue refund\n\nRules:\n- Keep responses brief and clear.\n- If a request is outside these three services, politely decline and restate supported services.\n- Never invent order_id or new_delivery_address.\n- Do not call tools until all required fields are explicitly provided by the customer.\n- For every new request, fetch parameter values from the customer again. Do not reuse values from prior turns.\n\nFor cancel order:\n- Required field: order_id\n- If order_id is missing, ask only for order_id.\n- If order_id is present, call cancel_order(order_id) directly.\n\nFor update delivery address:\n- Required fields: order_id and new_delivery_address\n- If any is missing, ask follow-up for missing fields only.\n- If both are present, call update_delivery_address(order_id, new_delivery_address) directly.\n\nFor issue re

## ASSIGNMENTS

1. The above experiment was executed over order cancellation. Try to continue the experiment over another customer request of supported type.

2. Instead of repeating multiple cell execution manually for each customer request, create a function that will take customer query as a parameter and output of the client. Provide the function an appropriate name.